# KNN-Based Book Recommendation System

* The following codes are written for the submission of the "[Book Recommendation Engine using KNN](https://www.freecodecamp.org/learn/machine-learning-with-python/machine-learning-with-python-projects/book-recommendation-engine-using-knn)" challenge as part of the "[Machine Learning with Python Certification](https://www.freecodecamp.org/learn/machine-learning-with-python/#tensorflow)" by freeCodeCamp. The challenge is deemed successful if the **`get_recommends` function accurately returns a list of 5 relevant and similar books with appropriate distance measures**.

* Kindly be notified that this notebook only serves as my personal notes referring to the materials provided in the course. All credit is attributable to the instructors of the courses.

* ChatGPT has also been used to provide understandable explanation for the steps done.

* A clean, honest walkthrough of everything I did and how each step affected my results is shown at the end of the notebook.

In [1]:
# import libraries 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
import html

from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors

## Load the datasets

In [2]:
# get data files
!wget https://cdn.freecodecamp.org/project-data/books/book-crossings.zip

!unzip book-crossings.zip

books_filename = 'BX-Books.csv'
ratings_filename = 'BX-Book-Ratings.csv'

--2026-04-16 04:07:27--  https://cdn.freecodecamp.org/project-data/books/book-crossings.zip
Resolving cdn.freecodecamp.org (cdn.freecodecamp.org)... 104.26.2.33, 104.26.3.33, 172.67.70.149, ...
Connecting to cdn.freecodecamp.org (cdn.freecodecamp.org)|104.26.2.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 26085508 (25M) [application/zip]
Saving to: 'book-crossings.zip'

book-crossings.zip  100%[===================>]  24.88M  --.-KB/s    in 0.09s   

2026-04-16 04:07:27 (274 MB/s) - 'book-crossings.zip' saved [26085508/26085508]

Archive:  book-crossings.zip
  inflating: BX-Book-Ratings.csv     
  inflating: BX-Books.csv            
  inflating: BX-Users.csv            


In [3]:
# import csv data into dataframes
df_books = pd.read_csv(
    books_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['isbn', 'title', 'author'],
    usecols=['isbn', 'title', 'author'],
    dtype={'isbn': 'str', 'title': 'str', 'author': 'str'})

df_ratings = pd.read_csv(
    ratings_filename,
    encoding = "ISO-8859-1",
    sep=";",
    header=0,
    names=['user', 'isbn', 'rating'],
    usecols=['user', 'isbn', 'rating'],
    dtype={'user': 'int32', 'isbn': 'str', 'rating': 'float32'})

## Explore the datasets

> The datasets contains the following features.

In [4]:
# Shape (rows, columns)
print("df_books shape:", df_books.shape)
print("df_ratings shape:", df_ratings.shape)

df_books shape: (271379, 3)
df_ratings shape: (1149780, 3)


In [5]:
# Column names
print("Columns:", df_books.columns.tolist())
print("Columns:", df_ratings.columns.tolist())

Columns: ['isbn', 'title', 'author']
Columns: ['user', 'isbn', 'rating']


In [6]:
# First few rows
df_books.head()

,isbn,title,author
0,0195153448,Classical Mythology,Mark P. O. Morford
1,0002005018,Clara Callan,Richard Bruce Wright
2,0060973129,Decision in Normandy,Carlo D'Este
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata
4,0393045218,The Mummies of Urumchi,E. J. W. Barber


In [7]:
df_ratings.head()

,user,isbn,rating
0,276725,034545104X,0.0
1,276726,0155061224,5.0
2,276727,0446520802,0.0
3,276729,052165615X,3.0
4,276729,0521795028,6.0


In [8]:
# Data types
print(df_books.dtypes)

isbn      object
title     object
author    object
dtype: object


In [9]:
print(df_ratings.dtypes)

user        int32
isbn       object
rating    float32
dtype: object


In [10]:
# Missing values
print(df_books.isnull().sum())

isbn      0
title     0
author    2
dtype: int64


In [11]:
print(df_ratings.isnull().sum())

user      0
isbn      0
rating    0
dtype: int64


In [12]:
# Unique counts
print("Unique ISBNs:", df_books['isbn'].nunique())
print("Unique titles:", df_books['title'].nunique())
print("Unique authors:", df_books['author'].nunique())
print("Unique users:", df_ratings['user'].nunique())
print("Unique ISBNs:", df_ratings['isbn'].nunique())

Unique ISBNs: 271379
Unique titles: 242154
Unique authors: 102041
Unique users: 105283
Unique ISBNs: 340556


In [13]:
# Most frequent titles (to see duplicates/editions)
df_books['title'].value_counts().head(10)

title
Selected Poems                    27
Little Women                      24
Wuthering Heights                 21
The Secret Garden                 20
Dracula                           20
Adventures of Huckleberry Finn    20
Jane Eyre                         19
The Night Before Christmas        18
Pride and Prejudice               18
Great Expectations                17
Name: count, dtype: int64

In [14]:
# Duplicate ISBNs (important)
duplicates = df_books['isbn'].duplicated().sum()
print("Duplicate ISBNs:", duplicates)

Duplicate ISBNs: 0


In [15]:
# Rating distribution
df_ratings['rating'].value_counts().sort_index()

rating
0.0     716109
1.0       1770
2.0       2759
3.0       5996
4.0       8904
5.0      50974
6.0      36924
7.0      76457
8.0     103736
9.0      67541
10.0     78610
Name: count, dtype: int64

In [16]:
# Check how many 0 ratings (VERY important)
print("Zero ratings:", (df_ratings['rating'] == 0).sum())

Zero ratings: 716109


> That’s ~62% of all data

In [17]:
# Ratings per user
user_counts = df_ratings['user'].value_counts()

print("Top users:")
print(user_counts.head())

print("\nUsers with >=200 ratings:", (user_counts >= 200).sum())

Top users:
user
11676     13602
198711     7550
153662     6109
98391      5891
35859      5850
Name: count, dtype: int64

Users with >=200 ratings: 905


In [18]:
# Ratings per book (ISBN)
book_counts = df_ratings['isbn'].value_counts()

print("Top books:")
print(book_counts.head())

print("\nBooks with >=100 ratings:", (book_counts >= 100).sum())

Top books:
isbn
0971880107    2502
0316666343    1295
0385504209     883
0060928336     732
0312195516     723
Name: count, dtype: int64

Books with >=100 ratings: 731


In [19]:
# Identify whether the 'user' column is user ID or the number of users
unique_users = df_ratings['user'].nunique()
total_rows = df_ratings.shape[0]

print(f"Unique users: {unique_users:,}")
print(f"Total rows: {total_rows:,}")

Unique users: 105,283
Total rows: 1,149,780


> The `user` column is definitely a user ID, not a count.

In [20]:
# Check overlap between tables
common_isbns = set(df_books['isbn']).intersection(set(df_ratings['isbn']))
print("Common ISBNs:", len(common_isbns))

Common ISBNs: 270170


## Feature Engineering for the Model

In [21]:
# Filter df_ratings by applying BOTH thresholds at the same time
df_ratings_clean = df_ratings[
    df_ratings['isbn'].isin(book_counts[book_counts >= 100].index) &
    df_ratings['user'].isin(user_counts[user_counts >= 200].index)
].copy()

print("cleaned ratings: ", df_ratings_clean.shape)

cleaned ratings:  (49781, 3)


In [22]:
# Merge cleaned ratings with original books
combined = pd.merge(df_ratings_clean, df_books, on='isbn', how='inner')

In [23]:
# Create pivot table
book_pivot = combined.pivot_table(
    index='title',
    columns='user',
    values='rating',
    fill_value=0
)

print("pivot: ", book_pivot.shape)

pivot:  (673, 888)


## Modelling

In [24]:
# Sparse matrix
book_matrix = csr_matrix(book_pivot.values)

In [25]:
# Train KNN
model = NearestNeighbors(metric='cosine', algorithm='brute')
model.fit(book_matrix)

NearestNeighbors(algorithm='brute', metric='cosine')

In [26]:
# Recommendation function
def get_recommends(book=""):
    if book not in book_pivot.index:
        return [book, []]

    book_idx = book_pivot.index.get_loc(book)

    distances, indices = model.kneighbors(
        book_matrix[book_idx],
        n_neighbors=6
    )

    recommended_books = []
    for i in range(1, len(distances.flatten())):
        recommended_books.append([
            book_pivot.index[indices.flatten()[i]],
            float(distances.flatten()[i])
        ])

    recommended_books.reverse()

    return [book, recommended_books]

In [27]:
books = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
print(books)

def test_book_recommendation():
  test_pass = True
  recommends = get_recommends("Where the Heart Is (Oprah's Book Club (Paperback))")
  if recommends[0] != "Where the Heart Is (Oprah's Book Club (Paperback))":
    test_pass = False
  recommended_books = ["I'll Be Seeing You", 'The Weight of Water', 'The Surgeon', 'I Know This Much Is True']
  recommended_books_dist = [0.8, 0.77, 0.77, 0.77]
  for i in range(2): 
    if recommends[1][i][0] not in recommended_books:
      test_pass = False
    if abs(recommends[1][i][1] - recommended_books_dist[i]) >= 0.05:
      test_pass = False
  if test_pass:
    print("You passed the challenge! 🎉🎉🎉🎉🎉")
  else:
    print("You haven't passed yet. Keep trying!")

test_book_recommendation()

["Where the Heart Is (Oprah's Book Club (Paperback))", [["I'll Be Seeing You", 0.8016210794448853], ['The Weight of Water', 0.7708583474159241], ['The Surgeon', 0.7699410915374756], ['I Know This Much Is True', 0.7677075266838074], ['The Lovely Bones: A Novel', 0.7234864234924316]]]
You passed the challenge! 🎉🎉🎉🎉🎉


## My Debugging Journey

**Here’s a clear, honest walkthrough of everything I did, what worked, what didn’t, and what I learned along the way.**

#### 1. Understanding the Problem

I started this project by building a book recommendation system using the Book-Crossing dataset. The goal was to use the **K-Nearest Neighbours (KNN)** algorithm to recommend similar books based on user ratings.

The dataset consisted of:

* `df_books`: ISBN, title, author
* `df_ratings`: ISBN, user, rating

The task was to:

* build a recommendation model using `NearestNeighbors`
* return 5 similar books for a given title
* match a very specific output format required by the test

---

#### 2. Initial Approach

My first approach was straightforward:

1. Merge `df_books` and `df_ratings`
2. Create a pivot table:
    * rows → book titles
    * columns → users
    * values → ratings
3. Train a KNN model using cosine similarity
4. Retrieve nearest neighbors for a given book

While this worked technically, the recommendations did not match the expected output.

---

#### 3. Debugging the Model Output

At first, I suspected issues with:

* feature engineering
* distance calculations
* model configuration

However, after testing, I confirmed that:

* KNN was working correctly
* cosine distance was appropriate
* the model produced reasonable recommendations

The issue was not with the algorithm itself.

---

#### 4. Investigating the Data

I then shifted focus to the dataset.

I explored:

* rating distributions
* number of users and books
* sparsity of the matrix

One important observation was:

* a large portion of ratings were `0`, representing implicit feedback

Although removing these seemed like a good idea conceptually, doing so broke the dataset under the challenge constraints, so I retained them.

---

#### 5. Understanding the Challenge Constraints

The challenge required:

* keeping users with at least 200 ratings
* keeping books with at least 100 ratings

A key realization was that:

* these filters must be applied directly on `df_ratings`
* both conditions must be enforced **simultaneously**, not sequentially

So instead of filtering after merging, I filtered `df_ratings` first based on:

* user frequency
* ISBN frequency

Then I merged the filtered ratings with `df_books`.

---

#### 6. The Critical Discovery

At one point, I followed advice from an online forum to remove duplicate book titles after merging:

> combined = combined.drop_duplicates(subset='title')

This seemed logical at first, but it turned out to be the **main reason my results did not match the expected output**.

Dropping duplicates:

* removed valid rating rows
* distorted the user-book interaction matrix
* weakened similarity signals

Once I **removed this line**, everything changed.

---

#### 7. Final Working Pipeline

The correct approach turned out to be:

1. Filter `df_ratings` using both thresholds:
    * users >= 200 ratings
    * books (ISBN) >= 100 ratings
2. Merge with `df_books`
3. Create pivot table:
    * index → title
    * columns → user
    * values → rating
4. Train KNN:
    * metric → cosine
    * algorithm → brute
5. Generate recommendations and reverse the list before returning

---

#### 8. Choice of Distance Metric and Algorithm

I chose **cosine similarity** and the **brute-force algorithm** for the KNN model based on the nature of the dataset.

* The book-user matrix is **high-dimensional** and **sparse** (many users, many missing ratings filled as 0).
* In such cases, cosine similarity works better than Euclidean distance because it measures the **angle between vectors**, focusing on similarity in rating patterns rather than magnitude.
* This makes it more suitable for recommendation systems where the pattern of user preferences matters more than absolute values.

For the algorithm:

* I used `algorithm='brute'` because tree-based methods like `kd_tree` and `ball_tree` are inefficient in **high-dimensional spaces**.
* Due to the **curse of dimensionality**, these methods lose their advantage and often perform worse than brute-force search.
* Brute-force, although simpler, works reliably with **sparse matrices and cosine distance**, which fits this problem well.

---

#### 9. Final Result

After fixing the duplicate issue, my model finally returned:

> ["Where the Heart Is (Oprah's Book Club (Paperback))",
 [
  ["I'll Be Seeing You", 0.80],
  ["The Weight of Water", 0.77],
  ["The Surgeon", 0.77],
  ["I Know This Much Is True", 0.76],
  ...
 ]]

And I successfully passed the challenge 🎉

---

#### 10. Key Lessons Learned

🔹 Data > Model

The issue was not the algorithm, but how the data was prepared.

🔹 Order of operations matters

Filtering before merging vs after merging produces very different results.

🔹 Small preprocessing steps can break everything

Dropping duplicate titles seemed harmless, but it completely changed the outcome.

🔹 Real-world vs test-case logic

Some steps that make sense in real-world systems (like deduplication) may not work for rigid test cases.

---